# 2일차 실습 P7 — 층마다 모양 찍기 · 원래 크기로 잘라내기

- **맨 위 준비 셀부터** 위에서 아래로 실행
- 셀 실행: 셀을 누르고 **Shift + Enter** 또는 셀 왼쪽 ▶
- Colab 에서 처음 열 때 경고 창이 뜨면 '계속' · 데이터는 내려받지 않음 · 준비 셀이 연습용 합성 문서를 만듦
- 모델 준비 셀: 작업 폴더에 `p_model_A.pt` 가 있으면 불러오고, 없으면 P6 과 같은 짝으로 10 에폭 학습함 · 학습하면 `맞춘 짝  학습 끝` 줄까지 기다림
- 빈칸은 `____` · 빈칸을 모두 바꾼 뒤 **그 셀부터 다시 실행**
- 막히면 `[안내]` 문장 → **에러 칸 맨 아래 줄** → 힌트 순서로 읽음
- 과제 셀에서 빨간 문법·실행 오류가 나면 확인 셀을 믿지 말 것 · 과제 셀을 고쳐 정상 실행한 뒤 확인 셀을 다시 실행할 것

## 에러를 읽는 법

- 에러 칸 첫 줄(`...Error    Traceback ...`)은 제목일 뿐 · 무엇이 틀렸는지는 **맨 아래 줄**
- 중간의 `---->` 화살표 줄 · torch 안쪽 칸은 처음엔 건너뜀 · **위에 찍힌 `[안내]` 문장과 맨 아래 줄부터** 읽음

| 맨 아래 줄에 보이는 말 | 뜻 | 먼저 볼 곳 |
|---|---|---|
| `name '____' is not defined` | 빈칸이 남음 | 화살표가 가리키는 줄 |
| `name 'model_A' is not defined` | 모델 준비 셀을 안 돌렸거나 런타임이 끊김 | 준비 셀 → 모델 준비 셀부터 다시 |
| `name 'dirty_b' is not defined` | P7-1 셀을 안 돌리고 P7-2 셀을 실행함 | P7-1 셀부터 |
| `too many indices for tensor of dimension 4` | 대괄호 안 칸 수가 모양 (판 수, 채널, 세로, 가로) 네 칸보다 많음 | ③ 줄의 대괄호 안 |

- **에러가 없는데** 확인 셀이 `[안내]` 를 찍으면 → 그 문장이 가리키는 줄부터
- 과제 셀에서 빨간 문법·실행 오류가 나면 확인 셀을 믿지 말 것 · 과제 셀을 고쳐 정상 실행한 뒤 확인 셀을 다시 실행할 것
- 아래 셀 맨 아래 줄이 `AssertionError: [안내] 확인 셀에서 …` → 확인 셀을 먼저 · P7-1 · P7-2 셀을 다시 실행했으면 확인 셀도 다시

## 준비

합성 문서 만들기 · 조각 자르기 · 모델 · 학습 함수 · 그림 함수 · 고치지 않고 실행

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

plt.rcParams.update({"figure.facecolor": "#1A222C", "axes.facecolor": "#1A222C", "savefig.facecolor": "#1A222C",   # 그림을 어두운 화면에 맞춤
                     "text.color": "#EEF2F6", "axes.labelcolor": "#EEF2F6", "xtick.color": "#EEF2F6", "ytick.color": "#EEF2F6", "axes.edgecolor": "#5C6A78"})

# ── 합성 문서 만들기 · 고치지 않고 실행 · 읽지 않아도 됨 ──
WORDS = ("the of and to in is was for on that with as by at from this be are or an report data model image paper result "
         "note page table value error clean noise sample layer filter record method letter number office meeting account "
         "review summary budget project quality section figure total").split()


def make_doc(seed, h=128, w=256, kinds=("gradient", "occlude", "saltpepper")):
    """문서 번호 하나 → (clean, dirty) · 값 0~1 · 같은 번호는 늘 같은 문서"""
    rng = np.random.default_rng(seed)
    size = int(rng.integers(12, 20))
    img = Image.new("L", (w, h), 255)
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=size)
    y = int(rng.integers(2, max(3, size // 2)))
    while y + int(size * 1.45) <= h - 2:                      # 줄마다 단어를 이어 씀
        x, words = int(rng.integers(2, 12)), []
        while True:
            cand = " ".join(words + [WORDS[int(rng.integers(len(WORDS)))]])
            if x + draw.textlength(cand, font=font) > w - 4:
                break
            words = cand.split()
        if words:
            draw.text((x, y), " ".join(words), fill=0, font=font)
        y += int(size * 1.45)
    clean = np.asarray(img, dtype=np.float32) / 255.0
    d = clean.copy()
    if "gradient" in kinds:                                   # 한쪽으로 갈수록 어두워짐
        g = np.linspace(0, rng.uniform(0.1, 0.35), w, dtype=np.float32)[None, :]
        if rng.random() < 0.5:
            g = g[:, ::-1]
        d = d * (1 - g)
    if "occlude" in kinds:                                    # 회색 상자가 글자를 가림
        oh, ow = int(h * rng.uniform(0.1, 0.25)), int(w * rng.uniform(0.05, 0.15))
        oy, ox = int(rng.integers(0, h - oh)), int(rng.integers(0, w - ow))
        d[oy:oy + oh, ox:ox + ow] = rng.uniform(0.2, 0.6)
    if "saltpepper" in kinds:                                 # 검은 점 · 흰 점
        m = rng.random(d.shape)
        p = rng.uniform(0.002, 0.01)
        d[m < p / 2] = 0.0
        d[m > 1 - p / 2] = 1.0
    return clean, np.clip(d, 0, 1).astype(np.float32)


# ── 오늘 쓰는 크기 · 문서 번호 · 도구 ──
H, W, PS = 128, 256, 48                 # 문서 세로 · 가로 · 조각 한 변
train_ids = list(range(1000, 1024))     # 학습 문서 24장
eval_ids = list(range(2000, 2008))      # 평가 문서 8장 · 학습에 쓰지 않음


def crop(img, y, x):                    # (y, x) 를 왼쪽 위 모서리로 삼아 48×48 조각을 잘라 냄
    return img[y:y + PS, x:x + PS]


def to_batch(patches):                  # 조각 목록 → (조각 수, 1, 48, 48) 텐서
    return torch.tensor(np.stack(patches)).unsqueeze(1)


def make_model():                       # 1일차 Conv 오토인코더와 같은 구성 · 48 → 24 → 12 → 24 → 48
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, 2, 1), nn.ReLU(),
        nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),
        nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1), nn.ReLU(),
        nn.ConvTranspose2d(16, 1, 3, 2, 1, output_padding=1), nn.Sigmoid())


def train(X, Y, name, epochs=10):       # 입력 X · 정답 Y 로 학습 · 에폭마다 loss 한 줄
    torch.manual_seed(0)                # 모델마다 같은 출발점
    model = make_model()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    t = time.time()
    for epoch in range(1, epochs + 1):
        perm = torch.randperm(len(X))
        total = 0.0
        for k in range(0, len(X), 32):
            b = perm[k:k + 32]
            opt.zero_grad()
            loss = F.mse_loss(model(X[b]), Y[b])
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        print(f"{name}  에폭 {epoch:>2}  loss {total / len(X):.4f}")
    print(f"{name}  학습 끝 ({time.time() - t:.0f}초)")
    return model


def show_rows(rows, idx):               # rows = [(줄 이름, 조각 묶음), …] · 줄마다 같은 번호의 조각
    fig, axes = plt.subplots(len(rows), len(idx), figsize=(len(idx), 1.25 * len(rows)))
    for r, (name, T) in enumerate(rows):
        for c, j in enumerate(idx):
            axes[r, c].imshow(T[j, 0], cmap="gray", vmin=0, vmax=1)
            axes[r, c].axis("off")
        axes[r, 0].set_title(name, loc="left", fontsize=10)
    plt.tight_layout()
    plt.show()

## 모델 준비

P6 과 같은 맞춘 짝 모델(48×48 조각 · 10 에폭) · 고치지 않고 실행

In [ ]:
# 맞춘 짝 모델 준비 — 저장 파일이 있으면 불러오고, 없으면 P6 과 같은 짝 · 같은 학습으로 만듦 · 고치지 않고 실행
import os

model_A = make_model()
if os.path.exists("p_model_A.pt"):
    model_A.load_state_dict(torch.load("p_model_A.pt"))
    print("맞춘 짝 모델 · 저장 파일 p_model_A.pt 를 불러옴")
else:
    rng = np.random.default_rng(2026)
    X, Y_A = [], []
    for i in train_ids:
        c, d = make_doc(i)
        ys, xs = rng.integers(0, H - PS + 1, 128), rng.integers(0, W - PS + 1, 128)
        for y, x in zip(ys, xs):
            X.append(crop(d, y, x))
            Y_A.append(crop(c, y, x))
    model_A = train(to_batch(X), to_batch(Y_A), "맞춘 짝")
model_A = model_A.eval()

## P7-1. 층마다 모양 찍기

1. 위 **준비** 셀 → **모델 준비** 셀을 실행
2. 아래 셀의 빈칸 ① 을 채움 · 반복 한 번에 층 하나씩 `t` 를 통과시키는 줄
3. 실행 → `입력` 한 줄 · 층마다 번호 · 이름 · 나온 모양이 한 줄씩 찍힘 · 이 표를 P7-2 에서 읽음
- 막히면 코드 셀 아래 **힌트 1** → **힌트 2** 순서로 하나씩 펼침

In [ ]:
# 문법 예시 (이 실습의 답 아님)
#   for k, name in enumerate(["a", "b"]):   → k 는 0, 1 · name 은 "a", "b" 를 차례로
#   act = nn.ReLU()  다음  z = act(z)        → 층 하나를 함수처럼 불러 z 를 통과시키고, 그 결과로 z 를 바꿈

trace = None                                     # 이 셀을 실행할 때마다 이전 결과를 지움 · 고치지 않음
모양_통과 = 대체_사용 = False                     # 확인 셀 통과 표시 · 대체 셀 사용 표시 · 고치지 않음

clean_b, dirty_b = make_doc(3000, h=130, w=270)  # 세로 130 · 가로 270 문서 한 장
trace = []                                       # (층 번호, 층 이름, 나온 모양) 기록
t = torch.tensor(dirty_b)[None, None]
print(f"입력                 {tuple(t.shape)}")
with torch.no_grad():
    for k, layer in enumerate(model_A):
        t = ____                                 # ① 층 하나 통과
        print(f"{k}번 {type(layer).__name__:<16} {tuple(t.shape)}")
        trace.append((k, type(layer).__name__, tuple(t.shape)))

<details><summary><b>힌트 1</b> — 막힐 때만 펼침</summary>

- ① 반복이 꺼내 주는 층 이름은 `layer` · 문법 예시의 `act` 처럼 함수로 부름 · 넣는 것은 지금까지 통과한 `t`

</details>

<details><summary><b>힌트 2</b> — 힌트 1 로도 막힐 때</summary>

- ① `t = layer(t)`

</details>

## P7-2. 처음 어긋나는 층 · 원래 크기로 잘라내기

1. 위 P7-1 에 찍힌 표를 읽음 · 한 줄 = 그 층을 **지나서 나온** 모양
2. 아래 셀의 빈칸 두 곳을 채움
   - ② 세로나 가로가 **홀수인 크기가 처음 들어가는** 칸 줄이기 층(`Conv2d`)의 번호 · 숫자 하나
   - ③ 4 의 배수로 채워 넣고 나온 `out` 에서 원래 문서 자리 `h × w` 만 잘라낸 것
3. 실행 → 채운 모양 · 나온 모양 · 잘라낸 모양 · 정답 모양이 찍힘 → 아래 **확인** 셀 실행 → `모양 확인 통과` 가 나오면 P7-3 으로 · **P7-1 · P7-2 셀을 다시 실행했으면 확인 셀도 다시**
- 확인 셀이 `[안내]` 를 찍으면 그 문장부터 읽음 · 막히면 코드 셀 아래 **힌트 1** → **힌트 2** 순서로 하나씩 펼침
- 그래도 막히면 확인 셀 아래 **대체 셀**을 실행하고 P7-3 으로

In [ ]:
# 문법 예시 (이 실습의 답 아님)
#   a = torch.zeros(1, 1, 6, 8)  다음  a[:, :, :3, :5]   → 앞 두 칸은 전부 · 세로 앞 3줄 · 가로 앞 5칸 → 모양 (1, 1, 3, 5)

first_odd_layer = out = out_crop = None          # 이 셀을 실행할 때마다 이전 결과를 지움 · 고치지 않음
모양_통과 = 대체_사용 = False                     # 확인 셀 통과 표시 · 대체 셀 사용 표시 · 고치지 않음

first_odd_layer = ____                           # ② 홀수 크기가 처음 들어가는 칸 줄이기 층(Conv2d)의 번호 · 숫자 하나

h, w = dirty_b.shape
H4, W4 = (h + 3) // 4 * 4, (w + 3) // 4 * 4      # 4 의 배수로 올림
padded = np.pad(dirty_b, ((0, H4 - h), (0, W4 - w)), mode="edge")   # 아래쪽 · 오른쪽에 가장자리 값을 늘여 채움
with torch.no_grad():
    out = model_A(torch.tensor(padded)[None, None])
out_crop = ____                                  # ③ out 에서 원래 문서 자리만 잘라냄
print("채운 모양", padded.shape, "  나온 모양", tuple(out.shape), "  잘라낸 모양", tuple(out_crop.shape), "  정답 모양", clean_b.shape)

<details><summary><b>힌트 1</b> — 막힐 때만 펼침</summary>

- ② k번 층에 **들어가는** 모양 = 표에서 바로 윗줄(0번은 `입력` 줄) · 칸을 줄이는 층은 `Conv2d` 줄 · 들어가는 세로 · 가로 가운데 홀수가 있는 첫 `Conv2d`
- ③ `out` 모양은 (판 수, 채널, 세로, 가로) · 채운 칸은 아래쪽 · 오른쪽 끝에 붙었음 · 원래 문서 자리는 왼쪽 위 `h` 줄 · `w` 칸 · 문법 예시와 같은 모양

</details>

<details><summary><b>힌트 2</b> — 힌트 1 로도 막힐 때</summary>

- ② `Conv2d` 줄은 0번 · 2번 · 0번에 들어가는 모양은 `입력` 줄 · 2번에 들어가는 모양은 1번 줄 · 둘 가운데 세로나 가로가 홀수인 쪽의 번호
- ③ `out[:, :, :h, :w]`

</details>

In [ ]:
# P7 확인 — 층 기록 · 층 번호 · 잘라낸 출력 검사 · 고치지 않고 실행만 · 읽지 않아도 됨
# 약속(수업 설명과 같음): 문서 3000 · 세로 130 · 가로 270 · 4 의 배수로 채울 때 아래쪽 · 오른쪽에 가장자리 값을 늘임 · 원래 문서 자리는 왼쪽 위
def 과제_문법_검사(*표시):                       # 과제 셀을 마지막으로 실행한 글자에 문법 오류가 있었는지 · 실행 기록(In)이 없으면 통과하지 않음
    기록 = globals().get("In")
    if not isinstance(기록, list) or len(기록) < 2:
        return False
    for 표 in 표시:
        for src in reversed(기록[:-1]):
            if 표 in src and "def 과제_문법_검사" not in src:
                try:
                    compile(src, "<과제 셀>", "exec")
                except SyntaxError:
                    return False
                break
    return True


def 모양_검사(trace, first_odd_layer, out_crop):
    if not trace:
        return "[안내] 위 P7-1 셀이 끝까지 실행되지 않았음 → 빈칸 ____ 을 바꾸고 P7-1 셀부터 다시 실행 · 준비 셀 · 모델 준비 셀을 안 돌렸으면 그것부터"
    c, d = make_doc(3000, h=130, w=270)          # 문서를 새로 만들어 비교
    t = torch.tensor(d)[None, None]
    기준, 처음 = [], None
    with torch.no_grad():
        for k in range(len(model_A)):
            층 = model_A[k]
            if 처음 is None and isinstance(층, nn.Conv2d) and (t.shape[-2] % 2 or t.shape[-1] % 2):
                처음 = k                          # 세로나 가로가 홀수인 크기가 처음 들어가는 칸 줄이기 층
            t = 층(t)
            기준.append((k, type(층).__name__, tuple(t.shape)))
    if len(trace) < len(기준):
        return "[안내] 위 P7-1 셀이 끝까지 실행되지 않았음 → 빈칸 ____ 을 바꾸고 P7-1 셀부터 다시 실행 · 준비 셀 · 모델 준비 셀을 안 돌렸으면 그것부터"
    if list(trace) != 기준:
        if all(tuple(s) == (1, 1, 130, 270) for _, _, s in trace):
            return "[안내] 모든 층에서 모양이 입력과 같음 → ① 줄은 t 를 이 층에 통과시킨 결과로 t 를 바꿈"
        return "[안내] 층마다 찍힌 모양이 실제 모델과 다름 → ① 줄에서 반복 한 번에 층 하나씩 통과시켰나"
    if first_odd_layer is None or out_crop is None:
        return "[안내] 위 P7-2 셀이 끝까지 실행되지 않았음 → 빈칸 ____ 두 곳을 모두 바꾸고 P7-2 셀부터 다시 실행"
    if isinstance(first_odd_layer, bool) or not isinstance(first_odd_layer, int):
        return "[안내] ② 에는 층 번호 숫자 하나 · 따옴표 없이"
    if first_odd_layer != 처음:
        if not 0 <= first_odd_layer < len(기준):
            return f"[안내] ② 층 번호는 0~{len(기준) - 1} 가운데 하나"
        종류 = 기준[first_odd_layer][1]
        if 종류 != "Conv2d":
            return f"[안내] {first_odd_layer}번은 {종류} · 칸을 줄이는 층이 아님 → 표에서 Conv2d 줄만 봄"
        return f"[안내] {first_odd_layer}번 Conv2d 에 들어가는 모양은 세로 · 가로 모두 짝수 → 표 한 줄 = 그 층을 지나서 나온 모양 · 들어가는 모양은 바로 윗줄"
    h, w = c.shape
    if not torch.is_tensor(out_crop) or tuple(out_crop.shape[-2:]) != (h, w) or out_crop.numel() != h * w:
        모양 = tuple(out_crop.shape) if hasattr(out_crop, "shape") else type(out_crop).__name__
        return f"[안내] 잘라낸 모양 {모양} 의 세로 · 가로가 정답 모양 ({h}, {w}) 과 다름 → 대괄호 안 순서 (판 수, 채널, 세로, 가로)"
    H4, W4 = (h + 3) // 4 * 4, (w + 3) // 4 * 4
    with torch.no_grad():
        채운_출력 = model_A(torch.tensor(np.pad(d, ((0, H4 - h), (0, W4 - w)), mode="edge"))[None, None])
    if not torch.equal(out_crop.detach().reshape(h, w), 채운_출력.narrow(2, 0, h).narrow(3, 0, w).reshape(h, w)):
        return "[안내] 모양은 맞지만 값이 '채워 넣고 나온 out 의 왼쪽 위' 와 다름 → out 에서 잘라냄 · 채운 칸은 아래쪽 · 오른쪽 끝에 붙었음"
    return "모양 확인 통과"


try:
    결과 = 모양_검사(globals().get("trace"), globals().get("first_odd_layer"), globals().get("out_crop")) if 과제_문법_검사('trace = None', 'first_odd_layer = out = out_crop = None') else "[안내] 과제 셀의 정상 실행을 확인할 수 없음(문법 오류 또는 실행 기록 없음) · 과제 셀을 고쳐 정상 실행한 뒤 확인 셀을 다시 실행 · 계속되면 런타임을 다시 시작하고 준비부터 실행"
except NameError:
    print("[안내] 준비 셀 · 모델 준비 셀을 안 돌렸음 → 맨 위 준비 셀부터 순서대로 다시 실행")
    raise
모양_통과 = 결과 == "모양 확인 통과"              # 통과를 찍을 때만 True · 아래 셀들이 이 값을 봄
모양_출처 = "대체 셀" if globals().get("대체_사용") else "직접 찍고 자름"
print(결과)

### 대체 셀 — 확인 셀이 `[안내]` 를 찍었고 막혔을 때만 실행

- P7-1 · P7-2 와 같은 층 기록 · 층 번호 · 잘라낸 출력을 다른 방법으로 만듦 · 실행하면 P7-3 으로 넘어갈 수 있음
- `모양 확인 통과` 가 나왔으면 실행하지 않음 · 실행하면 P7-3 첫 셀에 `모양: 대체 셀` 이 찍힘

In [ ]:
# 대체 셀 — 막혔을 때만 · P7-1 · P7-2 와 같은 trace · first_odd_layer · out · out_crop 을 다른 방법으로 만듦
clean_b, dirty_b = make_doc(3000, h=130, w=270)
t0 = torch.tensor(dirty_b)[None, None]
with torch.no_grad():
    들어간 = [tuple(model_A[:k](t0).shape) for k in range(len(model_A))]        # k번 층에 들어가는 모양 = 앞 k개 층을 지난 모양
    trace = [(k, type(model_A[k]).__name__, tuple(model_A[:k + 1](t0).shape)) for k in range(len(model_A))]
first_odd_layer = next(k for k, s in enumerate(들어간) if isinstance(model_A[k], nn.Conv2d) and (s[-2] % 2 or s[-1] % 2))
h, w = dirty_b.shape
H4, W4 = -(-h // 4) * 4, -(-w // 4) * 4
padded = np.pad(dirty_b, ((0, H4 - h), (0, W4 - w)), mode="edge")
with torch.no_grad():
    out = model_A(torch.tensor(padded)[None, None])
out_crop = out.narrow(2, 0, h).narrow(3, 0, w)
for k, name, s in trace:
    print(f"대체 셀 · {k}번 {name:<16} {s}")
print("대체 셀 · 잘라낸 모양", tuple(out_crop.shape), "  정답 모양", clean_b.shape)
모양_통과, 대체_사용, 모양_출처 = True, True, "대체 셀"   # 대체 셀로 만든 결과라는 표시 · P7-1 · P7-2 셀을 다시 실행하면 지워짐

## P7-3. 두 방법 나란히

아래 세 셀은 **고치지 않고** 차례로 실행 · 확인 셀 통과(또는 대체 셀) 뒤에만 돎

1. **방법 ②**: 48×48 조각으로 나눠 복원해 이어 붙임 · 첫 줄에 모양 출처
2. **그림**: clean · method 1(채우고 잘라냄 · 내가 자른 `out_crop`) · method 2(조각 이어 붙임) · 아래 줄은 조각 경계 부근 확대
3. **숫자**: 방법마다 조각 경계 부근 RMSE · 나머지 RMSE(문서 한 장 · 작을수록 정답에 가까움)
- 그림 → 숫자 순서로 본 뒤 맨 아래 마무리 칸에 고를 방법과 까닭을 적음

In [ ]:
# 방법 ② — 48×48 조각으로 나눠 복원해 이어 붙임 · 확인 셀 통과(또는 대체 셀) 뒤에만 · 고치지 않고 실행
assert globals().get("모양_통과"), "[안내] 확인 셀에서 '모양 확인 통과' 를 받은 뒤 실행 · P7-1 · P7-2 셀을 다시 실행했으면 확인 셀도 다시 · 막히면 대체 셀"
print("모양:", 모양_출처)
tiled = np.zeros_like(padded)
with torch.no_grad():
    for y in range(0, H4, PS):
        for x in range(0, W4, PS):
            part = padded[y:y + PS, x:x + PS]
            ph, pw = part.shape
            part4 = np.pad(part, ((0, (4 - ph % 4) % 4), (0, (4 - pw % 4) % 4)), mode="edge")
            tiled[y:y + ph, x:x + pw] = model_A(torch.tensor(part4)[None, None])[0, 0, :ph, :pw].numpy()
out_tile = tiled[:h, :w]
print("이어 붙인 모양", out_tile.shape)

In [ ]:
# 그림 — 정답 · 방법 ①(내가 자른 out_crop) · 방법 ② · 아래는 조각 경계 부근 확대
assert globals().get("모양_통과"), "[안내] 확인 셀에서 '모양 확인 통과' 를 받은 뒤 실행 · P7-1 · P7-2 셀을 다시 실행했으면 확인 셀도 다시 · 막히면 대체 셀"
crop_img = out_crop.reshape(h, w).numpy()           # 그림용 (130, 270)
fig, axes = plt.subplots(3, 1, figsize=(6, 6.4))
for ax, (name, img) in zip(axes, [("clean (answer)", clean_b), ("method 1: pad then crop", crop_img), ("method 2: tiles", out_tile)]):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name, loc="left", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

zy, zx = slice(20, 76), slice(20, 124)              # 가로 48 · 96 경계와 세로 48 경계가 들어간 부분
fig, axes = plt.subplots(1, 3, figsize=(9, 2.2))
for ax, (name, img) in zip(axes, [("clean", clean_b), ("method 1", crop_img), ("method 2", out_tile)]):
    ax.imshow(img[zy, zx], cmap="gray", vmin=0, vmax=1)
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 숫자 — 이 문서 한 장 · 조각 경계 4칸 부근과 나머지 · 작을수록 정답에 가까움
assert globals().get("모양_통과"), "[안내] 확인 셀에서 '모양 확인 통과' 를 받은 뒤 실행 · P7-1 · P7-2 셀을 다시 실행했으면 확인 셀도 다시 · 막히면 대체 셀"
seam = np.zeros((h, w), dtype=bool)
for y in range(PS, h, PS):
    seam[y - 2:y + 2] = True
for x in range(PS, w, PS):
    seam[:, x - 2:x + 2] = True
for name, img in [("방법 ① 채우고 잘라냄", crop_img), ("방법 ② 조각 이어 붙임", out_tile)]:
    err = (img - clean_b) ** 2
    print(f"{name}   경계 부근 RMSE {np.sqrt(err[seam].mean()):.3f}   나머지 RMSE {np.sqrt(err[~seam].mean()):.3f}   (문서 한 장)")

## P7 마무리 — 세 줄 적기 (채점 아님)

이 칸을 두 번 눌러 `→` 뒤에 적음

1. 두 방법(① 채우고 잘라냄 · ② 조각 이어 붙임) 가운데 고를 방법과 까닭 →
2. 그 까닭을 그림과 숫자에서 각각 무엇으로 봤나 →
3. 층마다 모양을 찍어 알게 된 것 한 줄 →
4. 대체 셀을 썼나 · 썼다면 어디서 막혔나 →